In [ ]:
# waypoint 가제보 시뮬레이션 테스트 파일
# ROS_DOMAIN_ID 확인
!echo $ROS_DOMAIN_ID

13


In [2]:
from nav2_simple_commander.robot_navigator import BasicNavigator
import rclpy

rclpy.init()
nav = BasicNavigator()

In [ ]:
nav.waitUntilNav2Active()

[INFO] [1762432671.206729153] [basic_navigator]: amcl/get_state service not available, waiting...
[INFO] [1762432672.208377434] [basic_navigator]: amcl/get_state service not available, waiting...
[INFO] [1762432673.211280205] [basic_navigator]: amcl/get_state service not available, waiting...
[INFO] [1762432674.213473797] [basic_navigator]: amcl/get_state service not available, waiting...
[INFO] [1762432675.216082474] [basic_navigator]: amcl/get_state service not available, waiting...
[INFO] [1762432676.218849463] [basic_navigator]: amcl/get_state service not available, waiting...
[INFO] [1762432677.220893742] [basic_navigator]: amcl/get_state service not available, waiting...
[INFO] [1762432678.224011005] [basic_navigator]: amcl/get_state service not available, waiting...
[INFO] [1762432679.229361674] [basic_navigator]: amcl/get_state service not available, waiting...
[INFO] [1762432680.232394749] [basic_navigator]: amcl/get_state service not available, waiting...
[INFO] [1762432681.2

In [4]:
# 9 로봇의 각도를 쿼터니언으로 변환 함수 선언
import math
import tf_transformations

def get_quaternion_from_yaw(yaw_degrees):
  yaw_radians = math.radians(yaw_degrees)
  
  quaternion = tf_transformations.quaternion_from_euler(0, 0, yaw_radians)
  
  return quaternion

In [5]:
# 11 현재 위치를 출력하기 위해 amcl_pose 토픽을 구독하는 노드 하나 만들기
sub_amcl = rclpy.create_node('sub_amcl')

In [6]:
def amcl_callback(msg):
  print('===')
  print(msg.pose.pose.position)
  print(msg.pose.pose.orientation)

In [7]:
from geometry_msgs.msg import PoseWithCovarianceStamped

sub_amcl.create_subscription(PoseWithCovarianceStamped, 'amcl_pose', amcl_callback, 10)

In [ ]:
# 12 제자리에서 회전하는 goal 생성
from geometry_msgs.msg import PoseStamped

goal_yaw = 270
q = get_quaternion_from_yaw(goal_yaw)

goal_pose = PoseStamped()
goal_pose.header.frame_id = 'map'
goal_pose.header.stamp = nav.get_clock().now().to_msg()
goal_pose.pose.position.x = 0.6
goal_pose.pose.position.y = -0.75
goal_pose.pose.position.z = 0.0
goal_pose.pose.orientation.x = q[0]
goal_pose.pose.orientation.y = q[1]
goal_pose.pose.orientation.z = q[2]
goal_pose.pose.orientation.w = q[3]

In [19]:
# 13 nav2에 goal 좌표 전송
nav.goToPose(goal_pose)

[INFO] [1762425808.114219595] [basic_navigator]: Navigating to goal: 0.6 -0.75...


True

In [20]:
# 15 goal 좌표 다시 보내기
from geometry_msgs.msg import PoseStamped

goal_yaw = 270
q = get_quaternion_from_yaw(goal_yaw)

goal_pose = PoseStamped()
goal_pose.header.frame_id = 'map'
goal_pose.header.stamp = nav.get_clock().now().to_msg()
goal_pose.pose.position.x = 0.6
goal_pose.pose.position.y = -2.35
goal_pose.pose.position.z = 0.0
goal_pose.pose.orientation.x = q[0]
goal_pose.pose.orientation.y = q[1]
goal_pose.pose.orientation.z = q[2]
goal_pose.pose.orientation.w = q[3]

In [21]:
nav.goToPose(goal_pose)

[INFO] [1762425854.939216825] [basic_navigator]: Navigating to goal: 0.6 -2.35...


True

In [26]:
# 18 주행 결과 확인
from nav2_simple_commander.robot_navigator import TaskResult

result = nav.getResult()
if result == TaskResult.SUCCEEDED:
  print('Goal succeeded!')
if result == TaskResult.CANCELED:
  print('Goal was canceled!')
elif result == TaskResult.FAILED:
  print('Goal failed!')
else:
  print('Goal has an invalid return status!')

Goal succeeded!
Goal has an invalid return status!


In [ ]:
# 19.4 waypoint list 만들기
goal_pose_list = []

def double_tuple_to_posestamped_goal_pose(frame_id, pose_x, pose_y, yaw_degree):
  q = get_quaternion_from_yaw(yaw_degree)
  goal_pose = PoseStamped()
  goal_pose.header.frame_id = 'map'
  goal_pose.header.stamp = nav.get_clock().now().to_msg()
  goal_pose.pose.position.x = pose_x
  goal_pose.pose.position.y = pose_y
  goal_pose.pose.orientation.x = q[0]
  goal_pose.pose.orientation.y = q[1]
  goal_pose.pose.orientation.z = q[2]
  goal_pose.pose.orientation.w = q[3]
  return goal_pose

goal_pose_list.append(double_tuple_to_posestamped_goal_pose('map', 0.6, -0.75, 270))
goal_pose_list.append(double_tuple_to_posestamped_goal_pose('map', 0.6, -2.35, 270))
goal_pose_list.append(double_tuple_to_posestamped_goal_pose('map', 1.6, -0.72, 270))

goal_pose_list

In [36]:
# 19.5 waypoint 주행 시작 후 피드백 확인
nav_start = nav.get_clock().now()
nav.followWaypoints(goal_pose_list)

[INFO] [1762430545.764117513] [basic_navigator]: Following 3 goals....


True